# Traffic Analytics System - Kaggle Training

This notebook trains your YOLO models on Kaggle GPUs (like Tesla T4).

### How to use this:
1. In Kaggle, click **File -> Import Notebook** and upload this `.ipynb` file.
2. On the right-hand panel, under **Accelerator**, select **GPU T4 x2**.
3. Run the cells below! It will automatically clone your code from GitHub and download all datasets.

## 1. Setup Environment
We clone your repository from GitHub and install the required libraries.

In [ ]:
# Clone your repository (Change this URL to match your repo)
!git clone https://github.com/YOUR_GITHUB_USERNAME/traffic-analytics-system.git /kaggle/working/traffic-analytics-system

import os
# Change directory to the project folder so our scripts work
if os.path.exists('/kaggle/working/traffic-analytics-system'):
    os.chdir('/kaggle/working/traffic-analytics-system')
else:
    print("Warning: traffic-analytics-system folder not found.")

!pip install -q ultralytics roboflow python-dotenv huggingface_hub pyyaml

## 2. Configure YOLO to save EVERY epoch
By default, YOLO only saves the `last.pt` and `best.pt` to save space. We will modify `configs/pipeline_config.yaml` to save every epoch (`save_period: 1`) and use dual GPUs if available.

In [ ]:
import yaml
import torch

config_path = 'configs/pipeline_config.yaml'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # Force save every 1 epoch
    config['training']['defaults']['save_period'] = 1
    
    # Force multi-GPU if 2 GPUs are found on Kaggle
    if torch.cuda.device_count() > 1:
        print(f"Found {torch.cuda.device_count()} GPUs, configuring for multi-GPU.")
        config['training']['device'] = '0,1'
    else:
        config['training']['device'] = 0

    with open(config_path, 'w') as f:
        yaml.dump(config, f, sort_keys=False)
        
    print("Updated config successfully. Models will save at every epoch.")
else:
    print(f"{config_path} not found. Make sure you extracted the project correctly.")

## 3. Download Datasets
We use your `download_datasets.py` script. The `--hf` flag ensures we can download from Hugging Face without needing API keys.

In [ ]:
!python download_datasets.py --all --hf

## 4. Train Models
All output, metrics (`results.csv`), and weights (`epoch1.pt`, `epoch2.pt`, `best.pt`, etc.) will be stored in `runs/detect/`.

In [ ]:
# Train Plate Detector
!python train_models.py --plates

In [ ]:
# Train Helmet Detector
!python train_models.py --helmets

In [ ]:
# Train Vehicle Detector
!python train_models.py --vehicles

## 5. Export Results
Kaggle automatically persists everything in `/kaggle/working`, but zipping the outputs makes it very easy to download them as a single file to your laptop.

In [ ]:
print("Zipping trained models and logs...")
!zip -r -q /kaggle/working/training_results.zip runs/ models/
print("Done! You can now download training_results.zip from the right-hand 'Output' panel.")